# Transformer_from_scratch (blank)

- 빈칸을 주석, 치트시트, 쿡북을 활용해 채워보시기 바랍니다.
- 빈칸 옆 주석은 **역할과 의도** 중심으로 작성되었습니다.
- 아래 **실습 코드**는 빈칸 없이 제공됩니다.

> 사용법  
> 1) 위에서부터 내려오며 `____`만 채우세요.  
> 2) 각 섹션의 체크 테스트를 수행해보세요.  
> 3) 마지막 “실습: Multi30k 학습” 섹션을 실행해보세요(시간이 오래 걸릴 수 있습니다).

빈칸은 **[N-1]~[N-36]의 36문장**입니다. 한 문장에 여러 `____`가 있어도 같은 번호입니다. 기본 Check와 개념 실험은 데이터 다운로드 없이 실행할 수 있고, Multi30k 학습은 선택입니다.

[Colab에서 열기](https://colab.research.google.com/github/aing-gachon/26-Spring-Transformer-Study/blob/main/Week2/A.ing_Transformer_from_scratch_blank.ipynb) · [치트시트](../Week1/A.ing_Transformer_Cheat_sheet.md) · [쿡북](A.ing_Transformer_Cookbook.md) · [논문 가이드](../Week1/transformer_paper_guide.md)

기반 코드: Aladdin Persson의 Transformer 교육 코드, A.ing 수정. 라이선스는 [README](../README.md)를 따릅니다.


## 학습 목표

1. **Scaled Dot-Product Attention**의 $\operatorname{softmax}(QK^T/\sqrt{d_k})V$를 구현하고 점수·가중치·출력을 구별할 수 있다.
2. **Multi-Head Attention**에서 `(N, L, E)` → `(N, L, h, d_k)`로 특징 축을 나누고 다시 합칠 수 있다.
3. **Residual·LayerNorm·FFN**에서 어떤 입력을 보존하고 어느 축을 변환하는지 설명할 수 있다.
4. **Encoder self-attention / Decoder masked self-attention / Cross-attention**의 Q·K·V 출처와 마스크를 구별할 수 있다.
5. 학습·평가에서 **target 한 칸 이동**, **PAD loss 제외**, `train() / eval() / no_grad()`의 역할을 구별할 수 있다.

> 허브: [R02](../Week1/transformer_paper_guide.md#r02) · [R03](../Week1/transformer_paper_guide.md#r03) · [R05](../Week1/transformer_paper_guide.md#r05) · [R06](../Week1/transformer_paper_guide.md#r06) · [R07](../Week1/transformer_paper_guide.md#r07) · [R08](../Week1/transformer_paper_guide.md#r08) · [R12](../Week1/transformer_paper_guide.md#r12)


## Self-Attention / Cross-Attention 용어 정리

같은 attention 계산이라도 **Q·K·V가 어디에서 왔는지**와 **어떤 위치를 허용하는지**가 다릅니다.

| 사용 위치 | Query 입력 | Key / Value 입력 | 사용하는 마스크 |
| --- | --- | --- | --- |
| Encoder self-attention | 현재 source 표현 | 현재 source 표현 | source padding mask |
| Decoder masked self-attention | 현재 target 표현 | 현재 target 표현 | target causal mask |
| Decoder cross-attention | masked self-attention 뒤 decoder 표현 | 최종 encoder 출력 | source padding mask |

이 노트북의 `SelfAttention`은 **세 경우에 공통으로 쓰는 직접 구현 클래스**입니다. 이름만 보고 항상 같은 문장에서 Q·K·V가 나온다고 생각하지 마세요. `forward(values, keys, query, mask)`에 무엇을 넣는지 확인합니다.

`N`은 배치 크기, `S`는 source 길이, `T`는 decoder 입력 길이, `E`는 모델 차원, `h`는 head 수입니다. **Source와 target 길이는 달라도 됩니다.** 마스크의 `True` 또는 `1`은 허용, `False` 또는 `0`은 차단을 뜻합니다.

> 허브: [R03](../Week1/transformer_paper_guide.md#r03) · [R06](../Week1/transformer_paper_guide.md#r06) · [R07](../Week1/transformer_paper_guide.md#r07)

## (선택) 실행을 위한 설치

이미 환경에 PyTorch가 있다면 건너뛰세요. 뒤의 Multi30k 학습에 필요한 패키지는 해당 실습 앞에서 별도로 설치합니다.


In [ ]:
%pip install torch


## 0) Transformer 핵심 ↔ Encoder–Decoder의 정보 흐름

- **전체 흐름**: source 토큰 → Encoder → source 표현 → Decoder → 다음 target 토큰의 점수
- **핵심 수식**: $\operatorname{Attention}(Q,K,V)=\operatorname{softmax}(QK^T/\sqrt{d_k})V$
- 직관: 각 query가 참고할 key에 가중치를 매기고, 그 위치의 value를 모아 새 표현을 만듭니다.

Colab에서 제한된 시간에 학습을 진행할 수 있도록 구성한 실습입니다. 학습형 위치 임베딩과 dropout 배치 등 실습 설정은 [논문 가이드](../Week1/transformer_paper_guide.md) 및 [쿡북 C5](A.ing_Transformer_Cookbook.md#c5)·[C13](A.ing_Transformer_Cookbook.md#c13)에서 확인하세요.

> 허브: [R02](../Week1/transformer_paper_guide.md#r02) · [R03](../Week1/transformer_paper_guide.md#r03)

## (준비) Import (모델 파트)

In [ ]:
import torch
import torch.nn as nn


### 🔧 진단 헬퍼 (그대로 실행)

Check 셀이 실패할 때 **어느 빈칸을 봐야 하는지**와 **쿡북 어느 절로 가야 하는지**를 알려 줍니다. 정답 대신 확인할 방향을 안내합니다. C 번호는 [쿡북](A.ing_Transformer_Cookbook.md)에서 찾으세요.


In [ ]:
def study_check_shape(actual, expected, where, route):
    actual, expected = tuple(actual), tuple(expected)
    if actual != expected:
        differences = [f"축 {i}: {a} != {b}" for i, (a, b) in enumerate(zip(actual, expected)) if a != b]
        if len(actual) != len(expected):
            differences.append(f"차원 수 {len(actual)} != {len(expected)}")
        raise AssertionError(f"{where}: 기대 {expected}, 실제 {actual}. " + "; ".join(differences) + f" → {route}")
    print(f"[OK] {where}: {actual}")


def study_check(condition, message, route):
    if not bool(condition):
        raise AssertionError(f"{message} → {route}")


def study_small_model(seed=7, layers=1):
    # 진단용 모델. 학습 데이터나 GPU 없이 실행합니다.
    torch.manual_seed(seed)
    return Transformer(24, 24, 0, 0, embed_size=32, num_layers=layers,
                       forward_expansion=2, heads=4, dropout=0.0,
                       device="cpu", max_length=16)


<a id="attention"></a>
## 1) Scaled Dot-Product + Multi-Head Attention ↔ `SelfAttention`

- **핵심 수식**: $\operatorname{softmax}(QK^T/\sqrt{d_k})V$
- 선형 변환 → head 분해 → query–key 점수 → mask → scaling·softmax → value 가중합 → head 결합 → 출력 변환 순서입니다.
- Q·K·V는 `(N, 길이, h, d_k)`, 점수표는 `(N, h, query_len, key_len)`입니다. 마지막 두 길이 축의 역할을 구별하세요.

### 사고 질문
- (why) 내적 점수에 $1/\sqrt{d_k}$ 스케일링을 넣는 이유는?
- (how) 여러 head가 서로 다른 관계를 학습할 수 있는 이유는 무엇일까요? 단일 head와 비교해 보세요.

> 허브: [R03](../Week1/transformer_paper_guide.md#r03) · [R04](../Week1/transformer_paper_guide.md#r04) · [R05](../Week1/transformer_paper_guide.md#r05) · [R07](../Week1/transformer_paper_guide.md#r07)
> 대응: [CS§1](../Week1/A.ing_Transformer_Cheat_sheet.md#cs1) · [CS§2](../Week1/A.ing_Transformer_Cheat_sheet.md#cs2) · [CS§4](../Week1/A.ing_Transformer_Cheat_sheet.md#cs4) · [C1](A.ing_Transformer_Cookbook.md#c1) · [C2](A.ing_Transformer_Cookbook.md#c2) · [C6](A.ing_Transformer_Cookbook.md#c6) · [C7](A.ing_Transformer_Cookbook.md#c7) · [C8](A.ing_Transformer_Cookbook.md#c8) · [C9](A.ing_Transformer_Cookbook.md#c9) · [C10](A.ing_Transformer_Cookbook.md#c10) · [C11](A.ing_Transformer_Cookbook.md#c11) · [C12](A.ing_Transformer_Cookbook.md#c12)


In [ ]:
# =========================================================
# 1) SelfAttention (Scaled Dot-Product + Multi-Head)
# - 쿡북: [C1-1] · [C2-1] · [C6-1] · [C7-1] · [C8-1] · [C9-1] · [C10-1] · [C11-1] · [C12-1]
# - Eq.(1): softmax(QK^T / sqrt(d_k)) V
# =========================================================

class SelfAttention(nn.Module):
    def __init__(self, embed_size, heads):
        super(SelfAttention, self).__init__()
        self.d_model = embed_size
        self.h = heads
        # [CS§2 Multi-Head Attention | Step 1/2]
        # 힌트: A.ing_Transformer_Cookbook.md [C2-1]의 코드 사용법과 Shape 흐름을 먼저 확인하세요.
        self.d_k = self.d_model // _____________  # [N-1] 모델 차원을 head 수로 나누어 head 하나의 특징 수를 구한다 → [R05] [C2-1] [CS§2]
        # [CS§2 Multi-Head Attention | Step 2/2]
        assert (
            self.h * self.d_k == self.d_model
        ), "Embedding size needs to be divisible by heads"

        # [CS§1 Scaled Dot-Product Attention | Step 1/2] Q,K,V 선형변환 정의 (Eq.1)
        # 힌트: A.ing_Transformer_Cookbook.md [C6-1]의 코드 사용법과 Shape 흐름을 먼저 확인하세요.
        # ================================================================================
        # Multi-Head Attention에서는 각 head가 Q, K, V를 (d_model -> d_k)로 변환합니다.
        # 논문의 base 설정은 head가 8개입니다. 이 노트북에서는 heads 설정값만큼의 변환을 한 번에 계산합니다.
        # d_model -> d_model 선형 변환을 한 뒤 특징 축을 head 수만큼 나누면 각 head의 변환을 묶어서 계산할 수 있습니다.
        # 이 말이 이해 되셨다면 빈칸을 채우실 수 있습니다!
        # ================================================================================
        self.W_V = nn.Linear(__________, ____________)  # [N-2] value 입력을 모델 차원의 벡터로 변환하는 층을 만든다 → [R03] [C6-1] [CS§1]
        self.W_K = nn.Linear(__________, ____________)  # [N-3] key 입력을 모델 차원의 벡터로 변환하는 층을 만든다 → [R03] [C6-1] [CS§1]
        self.W_Q = nn.Linear(__________, ____________)  # [N-4] query 입력을 모델 차원의 벡터로 변환하는 층을 만든다 → [R03] [C6-1] [CS§1]
        # [CS§2 Multi-Head Attention | Step 2/2] Concat(heads) 이후 W^O (Fig.2)
        # 힌트: A.ing_Transformer_Cookbook.md [C12-1]의 코드 사용법과 Shape 흐름을 먼저 확인하세요.
        self.W_O = nn.Linear(__________, ____________)  # [N-5] 여러 head를 결합한 특징을 다시 조합하는 출력 층을 만든다 → [R05] [C12-1] [CS§2]

    def forward(self, values, keys, query, mask):
        # [CS§6 토큰·임베딩 shape | Step 1/9] 배치 크기 N / 길이 추출
        # ================================================================================
        # NLP 모델에서 텐서의 shape은 보통 (배치 크기, 문장 길이, 임베딩 차원) 순서로 들어옵니다.
        # CS§6 토큰·임베딩 shape을 보셨다면 N이 무엇을 의미하는지 아실 겁니다.
        # ================================================================================
        N = ____________  # [N-6] query 텐서에서 배치 크기를 읽는다 → [R02] [C1-1] [CS§6]

        value_len, key_len, query_len = values.shape[1], keys.shape[1], query.shape[1]

        # [CS§1 Scaled Dot-Product Attention | Step 2/9] Q, K, V 만들기 (Eq.1)
        # 힌트: A.ing_Transformer_Cookbook.md [C6-1]의 코드 사용법과 Shape 흐름을 먼저 확인하세요.
        V = self.W_V(values)
        K = self.W_K(keys)
        Q = self.W_Q(query)

        # [CS§2 Multi-Head Attention | Step 3/9]
        # 힌트: A.ing_Transformer_Cookbook.md [C7-1]의 코드 사용법과 Shape 흐름을 먼저 확인하세요.
        # ================================================================================
        # 여기서는 Multi Head Attention을 위해 Q, K, V를 head 수만큼 나누는 과정입니다.
        # 기존의 Q, K, V의 shape은 (배치 크기(N), 문장 길이(len), d_model)입니다.
        # 여기서 head와 관련된 연산이 수행된 변수가 하나 있습니다.
        # 이제 빈칸을 풀어보세요!
        # ================================================================================
        V = V.reshape(N, value_len, ____________, ____________)  # [N-7] value의 특징 축을 head 수와 head 차원으로 나눈다 → [R05] [C7-1] [CS§2]
        K = K.reshape(N, key_len, ____________, ____________)  # [N-8] key의 특징 축을 head 수와 head 차원으로 나눈다 → [R05] [C7-1] [CS§2]
        Q = Q.reshape(N, query_len, ____________, ____________)  # [N-9] query의 특징 축을 head 수와 head 차원으로 나눈다 → [R05] [C7-1] [CS§2]

        # [CS§1 Scaled Dot-Product Attention | Step 4/9] attention_logits = QK^T (Eq.1)
        # 힌트: A.ing_Transformer_Cookbook.md [C8-1]의 코드 사용법과 Shape 흐름을 먼저 확인하세요.
        attention_logits_QK = torch.einsum("____________________", [Q, K])  # [N-10] head마다 모든 query–key 쌍의 내적 점수를 구한다 → [R03] [C8-1] [CS§1]

        # [CS§4 Mask | Step 5/9] mask 적용 (padding/causal 차단)
        # 힌트: A.ing_Transformer_Cookbook.md [C9-1]의 코드 사용법과 Shape 흐름을 먼저 확인하세요.
        if mask is not None:
            attention_logits_QK = attention_logits_QK.masked_fill(__________ == 0, float("-1e20"))  # [N-11] 허용 마스크에서 0인 위치의 점수를 매우 작은 값으로 바꾼다 → [R07] [C9-1] [CS§4]

        # [CS§1 Scaled Dot-Product Attention | Step 6/9] scale(+softmax) (Eq.1)
        # - / sqrt(d_k) 로 softmax 포화 방지
        # - softmax dim은 key_len 축(마지막 축)
        # 힌트: A.ing_Transformer_Cookbook.md [C10-1]의 코드 사용법과 Shape 흐름을 먼저 확인하세요.
        attention_weights = torch.softmax(attention_logits_QK / (self.d_k ** (1 / 2)), dim=-1)

        # [CS§1 Scaled Dot-Product Attention | Step 7/9] out_heads = attention_weights @ V (Eq.1)
        # 힌트: A.ing_Transformer_Cookbook.md [C11-1]의 코드 사용법과 Shape 흐름을 먼저 확인하세요.
        out_heads = torch.einsum("____________________", [attention_weights, V])  # [N-12] key 위치를 따라 value를 가중합해 query별 출력을 만든다 → [R03] [C11-1] [CS§1]
        # [CS§2 Multi-Head Attention | Step 8/9]
        out = out_heads.reshape(N, query_len, ____________ * ____________)  # [N-13] query 길이를 유지한 채 마지막 두 head 관련 축을 합친다 → [R05] [C12-1] [CS§2]

        # [CS§2 Multi-Head Attention | Step 9/9] W^O output projection (Fig.2)
        out = self.W_O(out)

        return out


### ✅ Check 1: SelfAttention shape 테스트
(빈칸을 채운 뒤 실행)

각 query에서 최소 하나의 key를 볼 수 있어야 합니다. 마스크 적용 뒤에도 출력 길이는 query 길이를 따릅니다.

> **실행 전에 한 번**: 출력 shape이 얼마일지 먼저 적어 보세요 — `(  ,  ,  )`.
> 틀렸다면 어디서 어긋났는지가 곧 배울 지점입니다.

> 허브: [R03](../Week1/transformer_paper_guide.md#r03) · [R05](../Week1/transformer_paper_guide.md#r05) · [R07](../Week1/transformer_paper_guide.md#r07)
> 쿡북: [C7](A.ing_Transformer_Cookbook.md#c7) · [C8](A.ing_Transformer_Cookbook.md#c8) · [C9](A.ing_Transformer_Cookbook.md#c9) · [C10](A.ing_Transformer_Cookbook.md#c10) · [C11](A.ing_Transformer_Cookbook.md#c11) · [C12](A.ing_Transformer_Cookbook.md#c12)


In [ ]:
import torch

torch.manual_seed(0)

N = 2
q_len = 5
d_model = 32
h = 4

self_attn = SelfAttention(embed_size=d_model, heads=h)

query = torch.randn(N, q_len, d_model)  # (N, q_len, d_model)
keys = query  # (N, q_len, d_model)
values = query  # (N, q_len, d_model)

mask = torch.ones(N, 1, 1, q_len)  # (N, 1, 1, k_len)
mask[0, :, :, -1] = 0  # (N, 1, 1, k_len)

out = self_attn(values=values, keys=keys, query=query, mask=mask)  # (N, q_len, d_model)
study_check_shape(out.shape, (N, q_len, d_model), "out", "C7–C12 / N-7–N-13")
assert torch.isfinite(out).all(), "NaN/Inf detected in SelfAttention output"
print("[OK] SelfAttention output shape:", tuple(out.shape))


### ✅ Check 1.5: Cross-attention 값·길이·padding 테스트
(빈칸을 채운 뒤 실행)

Query 길이 3, key/value 길이 5입니다. 가린 key/value의 내용만 바꾸면 출력이 달라져야 할까요? 출력 shape와 함께 PAD 정보 차단·허용 value의 영향·head별 계산을 검사합니다. 별도 PyTorch attention API를 기준으로 값을 비교합니다.

> **실행 전에 한 번**: 출력 shape이 얼마일지 먼저 적어 보세요 — `(  ,  ,  )`.
> 틀렸다면 어디서 어긋났는지가 곧 배울 지점입니다.

> 허브: [R03](../Week1/transformer_paper_guide.md#r03) · [R05](../Week1/transformer_paper_guide.md#r05) · [R07](../Week1/transformer_paper_guide.md#r07)
> 쿡북: [C6](A.ing_Transformer_Cookbook.md#c6) · [C7](A.ing_Transformer_Cookbook.md#c7) · [C8](A.ing_Transformer_Cookbook.md#c8) · [C9](A.ing_Transformer_Cookbook.md#c9) · [C10](A.ing_Transformer_Cookbook.md#c10) · [C11](A.ing_Transformer_Cookbook.md#c11) · [C12](A.ing_Transformer_Cookbook.md#c12)


In [ ]:
import torch.nn.functional as F

torch.manual_seed(11)
attn = SelfAttention(embed_size=32, heads=4).eval()
q = torch.randn(2, 3, 32)
k = torch.randn(2, 5, 32)
v = torch.randn(2, 5, 32)
visible = torch.ones(2, 1, 1, 5, dtype=torch.bool)
visible[..., -1] = False
with torch.no_grad():
    actual = attn(v, k, q, visible)
    study_check_shape(actual.shape, (2, 3, 32), "cross-attention", "C12/C17 / N-13/N-22")
    k_changed, v_changed = k.clone(), v.clone()
    k_changed[:, -1] += 100
    v_changed[:, -1] -= 100
    masked_change = attn(v_changed, k_changed, q, visible)
    study_check(torch.allclose(actual, masked_change, atol=1e-5),
                "차단한 key/value의 내용이 출력에 섞였습니다", "C3/C9 / N-11")
    active_v = v.clone()
    active_v[:, 0] += 10
    study_check(not torch.allclose(actual, attn(active_v, k, q, visible), atol=1e-5),
                "허용한 value를 바꿔도 출력이 같습니다", "C11/C12 / N-12/N-13")
    # 별도 API를 기준으로 projection부터 head 결합까지 비교합니다.
    def split_reference(layer, x):
        return layer(x).unflatten(-1, (4, 8)).transpose(1, 2)
    expected = F.scaled_dot_product_attention(
        split_reference(attn.W_Q, q), split_reference(attn.W_K, k),
        split_reference(attn.W_V, v), attn_mask=visible, dropout_p=0.0)
    expected = attn.W_O(expected.transpose(1, 2).flatten(-2))
    study_check(torch.allclose(actual, expected, atol=1e-5, rtol=1e-4),
                "독립 attention 기준과 값이 다릅니다", "C6–C12 / N-2–N-13")
print("[OK] Cross-attention 값·길이·padding 검사")


<a id="block"></a>
## 2) Add & Norm + FFN ↔ `TransformerBlock.forward`

이 실습의 블록은 아래 흐름입니다.

1) Multi-Head Attention  
2) query와 residual 덧셈 → LayerNorm → Dropout  
3) FFN: 선형 확장 → ReLU → 선형 축소  
4) FFN 입력과 residual 덧셈 → LayerNorm → Dropout

**CheatSheet 대응 포인트**
- 두 번의 residual 덧셈에서 보존할 입력이 각각 무엇인지 확인하세요.
- FFN은 위치 사이를 섞지 않고 각 위치의 특징을 변환합니다.
- Dropout은 이 노트북에서 정규화 뒤에 적용합니다. 원논문의 dropout 위치와는 차이가 있습니다.

### 사고 질문
- (why) Attention 뒤에 바로 FFN을 한 번 더 넣는 이유는?
- (how) residual이 없으면 역전파에서 어떤 문제가 생길까?

> 허브: [R08](../Week1/transformer_paper_guide.md#r08)
> 대응: [CS§5](../Week1/A.ing_Transformer_Cheat_sheet.md#cs5) · [C13](A.ing_Transformer_Cookbook.md#c13) · [C14](A.ing_Transformer_Cookbook.md#c14)


In [ ]:
# =========================================================
# 2) TransformerBlock (Add & Norm + FFN)
# - 쿡북: [C13-1] · [C13-2] · [C14-1] · [C14-2]
# =========================================================

class TransformerBlock(nn.Module):
    def __init__(self, embed_size, heads, dropout, forward_expansion):
        super(TransformerBlock, self).__init__()
        self.attention = SelfAttention(embed_size, heads)
        self.norm1 = nn.LayerNorm(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)

        # [CS§5 Add & Norm + FFN | Step 1/2]
        # 힌트: A.ing_Transformer_Cookbook.md [C14-1]의 코드 사용법과 Shape 흐름을 먼저 확인하세요.
        self.feed_forward = nn.Sequential(
            nn.Linear(embed_size, forward_expansion * embed_size),
            nn.ReLU(),
            nn.Linear(forward_expansion * embed_size, embed_size),
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, value, key, query, mask):
        # [CS§5 Add & Norm + FFN | Step 1/4] (Sublayer) Multi-Head Attention
        # 힌트: A.ing_Transformer_Cookbook.md [C13-1]의 코드 사용법과 Shape 흐름을 먼저 확인하세요.
        multihead_attention_output = self.attention(value, key, query, mask)

        # [CS§5 Add & Norm + FFN | Step 2/4] Add & Norm (Residual + LayerNorm + Dropout)
        # ================================================================================
        # Residual은 입력을 더해 정보가 흐를 통로를 만듭니다.
        # 쿡북 [C13-1]·[C14-2]에서 residual 덧셈의 두 입력과 shape을 확인하세요.
        # 현재 실습은 덧셈 → LayerNorm → Dropout 순서이며, 원논문의 dropout 위치와는 차이가 있습니다.
        # 현재 단계는 query에 대해 가장 관련있는 value를 찾는 과정이면 attention 결과에 무엇을 더해야 할지 아시겠죠?
        # ================================================================================
        attention_residual_add = ____________ + ____________  # [N-14] attention 결과에 그 attention으로 들어간 query 표현을 더한다 → [R08] [C13-1] [CS§5]
        post_attention_layernorm = self.norm1(attention_residual_add)
        x = self.dropout(post_attention_layernorm)

        # [CS§5 Add & Norm + FFN | Step 3/4] Position-wise FFN (shape를 보이게 쪼개서 실행)
        # 힌트: A.ing_Transformer_Cookbook.md [C14-1]의 코드 사용법과 Shape 흐름을 먼저 확인하세요.
        ffn_linear1_output = self.feed_forward[0](x)
        ffn_relu_output = self.feed_forward[1](ffn_linear1_output)
        ffn_linear2_output = self.feed_forward[2](ffn_relu_output)

        # [CS§5 Add & Norm + FFN | Step 4/4] Add & Norm after FFN
        # ================================================================================
        # Residual은 입력을 더해 정보가 흐를 통로를 만듭니다.
        # 쿡북 [C13-1]·[C14-2]에서 residual 덧셈의 두 입력과 shape을 확인하세요.
        # 현재 실습은 덧셈 → LayerNorm → Dropout 순서이며, 원논문의 dropout 위치와는 차이가 있습니다.
        # 현재 단계는 x에 대해 FFN 결과를 출력하는 단계이니 skip connection을 구상할 아이디어가 떠오르시죠?
        # ================================================================================
        ffn_residual_add = ____________ + ____________  # [N-15] FFN 결과에 FFN 직전의 입력 표현을 더한다 → [R08] [C14-2] [CS§5]
        post_ffn_layernorm = self.norm2(ffn_residual_add)
        out = self.dropout(post_ffn_layernorm)

        return out


### ✅ Check 2: TransformerBlock shape 테스트
(빈칸을 채운 뒤 실행)

Attention과 FFN 뒤에도 위치 수와 모델 차원이 유지되는지 확인합니다.

> **실행 전에 한 번**: 출력 shape이 얼마일지 먼저 적어 보세요 — `(  ,  ,  )`.
> 틀렸다면 어디서 어긋났는지가 곧 배울 지점입니다.

> 허브: [R08](../Week1/transformer_paper_guide.md#r08)
> 쿡북: [C13](A.ing_Transformer_Cookbook.md#c13) · [C14](A.ing_Transformer_Cookbook.md#c14)


In [ ]:
import torch

torch.manual_seed(0)

N = 2
seq_len = 6
d_model = 32
h = 4

block = TransformerBlock(embed_size=d_model, heads=h, dropout=0.0, forward_expansion=4)

x = torch.randn(N, seq_len, d_model)  # (N, seq_len, d_model)
mask = torch.ones(N, 1, 1, seq_len)  # (N, 1, 1, seq_len)

out = block(value=x, key=x, query=x, mask=mask)  # (N, seq_len, d_model)
study_check_shape(out.shape, (N, seq_len, d_model), "out", "C13–C14 / N-14–N-15")
assert torch.isfinite(out).all(), "NaN/Inf detected in TransformerBlock output"
print("[OK] TransformerBlock output shape:", tuple(out.shape))


<a id="encoder"></a>
## 3) Encoder ↔ `Encoder.forward`

- 토큰 임베딩과 학습형 위치 임베딩을 더합니다.
- Encoder 블록마다 현재 source 표현을 query·key·value 입력으로 전달합니다.
- Source padding mask는 모든 Encoder 층에서 유지합니다.
- 출력은 `(N, S, E)`입니다. 위치 임베딩은 원논문의 sinusoidal 방식 대신 학습형 표를 사용합니다.

### 사고 질문
- (why) 순서 정보가 없으면 Self-Attention은 어떤 문제가 생길까?
- (how) 학습형 position embedding과 sinusoidal의 장단점은?

> 허브: [R06](../Week1/transformer_paper_guide.md#r06) · [R09](../Week1/transformer_paper_guide.md#r09) · [R10](../Week1/transformer_paper_guide.md#r10)
> 대응: [CS§3](../Week1/A.ing_Transformer_Cheat_sheet.md#cs3) · [CS§6](../Week1/A.ing_Transformer_Cheat_sheet.md#cs6) · [C4](A.ing_Transformer_Cookbook.md#c4) · [C5](A.ing_Transformer_Cookbook.md#c5) · [C15](A.ing_Transformer_Cookbook.md#c15)


In [ ]:
# =========================================================
# 3) Encoder (Embedding + Positional Encoding + Encoder Stack)
# - 쿡북: [C4-1] · [C5-1] · [C15-1]
# =========================================================

class Encoder(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        embed_size,
        num_layers,
        heads,
        device,
        forward_expansion,
        dropout,
        max_length,
    ):

        super(Encoder, self).__init__()
        self.embed_size = embed_size
        self.device = device
        self.word_embedding = nn.Embedding(src_vocab_size, embed_size)
        self.position_embedding = nn.Embedding(max_length, embed_size)

        self.layers = nn.ModuleList(
            [
                TransformerBlock(
                    embed_size,
                    heads,
                    dropout=dropout,
                    forward_expansion=forward_expansion,
                )
                for _ in range(num_layers)
            ]
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, src_token_ids, src_padding_mask):
        # [CS§6 토큰·임베딩 shape | Step 1/3] src_token_ids: (N, src_len)
        # 힌트: A.ing_Transformer_Cookbook.md [C1-1]의 코드 사용법과 Shape 흐름을 먼저 확인하세요.
        N, src_len = src_token_ids.shape

        # [CS§6 Embedding + Positional Encoding | Step 2/3] positions 만들기
        # 힌트: A.ing_Transformer_Cookbook.md [C5-1]의 코드 사용법과 Shape 흐름을 먼저 확인하세요.
        # ================================================================================
        # Positional Encoding을 알고 있다면 "위치 인덱스 만들기:"부터 읽으셔도 됩니다.
        # RNN은 이전 위치의 상태를 다음 위치로 전달하므로 순서가 계산 경로에 반영됩니다. 멀리 떨어진 위치의 정보는 여러 단계를 거쳐야 합니다.
        # Transformer의 self-attention은 여러 위치의 관계를 직접 계산하고, 학습 시 위치별 계산을 병렬로 처리할 수 있습니다. 대신 위치 정보를 별도로 전달해야 합니다.
        # 위치 정보가 없는 self-attention은 입력 순서를 바꾸면 출력도 같은 순서로 바뀝니다. E02에서 이 성질을 확인합니다. 따라서 문장의 "토큰"마다 고유한 위치정보를 더해야 합니다.

        # 위치 인덱스 만들기:
        # 1. arange(0, x)를 사용해 [0, 1, 2, ..., "문장길이"-1] 형태의 1차원 번호표를 만듭니다.
        # 2. expand(N, L)는 같은 위치 번호를 N개 문장에 broadcast합니다. 실제 데이터를 복사하지 않습니다.
        # 즉, (L,)을 (N, L)로 확장합니다.
        # 그렇다면 빈칸을 어떻게 채워야 할까요?
        # ================================================================================

        position_ids = torch.arange(0, ____________).to(self.device)  # [N-16] source 길이만큼 위치 번호를 만든다 → [R10] [C5-1] [CS§6]
        positions = position_ids.expand(__________, ____________)  # [N-17] source 위치 번호를 배치 전체에 공유한다 → [R10] [C5-1] [CS§6]

        # [CS§6 Embedding + Positional Encoding | Step 3/3] token_emb + pos_emb (+ dropout)
        # 힌트: A.ing_Transformer_Cookbook.md [C4-1] · [C5-1]
        token_embedding_d_model = self.word_embedding(src_token_ids)
        positional_embedding_d_model = self.position_embedding(positions)
        out = self.dropout(____________ + ____________)  # [N-18] source의 토큰 벡터와 위치 벡터를 더한다 → [R10] [C5-1] [CS§6]


        # [CS§3 Encoder Self-Attention | Step 1/1] Encoder stack 반복
        # 힌트: A.ing_Transformer_Cookbook.md [C15-1]의 코드 사용법과 Shape 흐름을 먼저 확인하세요.
        # ================================================================================
        # 여기서 layer는 self.layers의 TransformerBlock 객체입니다.
        # nn.Module 객체를 layer(...)처럼 호출하면 내부의 forward가 실행됩니다. 객체를 생성하거나 변수에서 읽는 것만으로 실행되는 것은 아닙니다.
        # 위의 두 문장을 연결해서 생각해보세요. 그러면 아래 4개의 입력은 Transformer의 어떤 함수를 실행시키기 위해 필요할까요?
        # 질문에 답을 할 수 있다면 빈칸을 채우실 수 있습니다!
        # ================================================================================
        for layer in self.layers:
            out = layer(____________, ____________, ____________, ____________)  # [N-19] 현재 source 표현과 source 마스크를 다음 Encoder 블록에 전달한다 → [R06] [C15-1] [CS§3]

        return out


### ✅ Check 3: Encoder shape 테스트
(빈칸을 채운 뒤 실행)

토큰 ID의 2차원 입력이 Encoder를 지난 뒤 어떤 특징 축을 가지는지 확인합니다.

> **실행 전에 한 번**: 출력 shape이 얼마일지 먼저 적어 보세요 — `(  ,  ,  )`.
> 틀렸다면 어디서 어긋났는지가 곧 배울 지점입니다.

> 허브: [R06](../Week1/transformer_paper_guide.md#r06) · [R10](../Week1/transformer_paper_guide.md#r10)
> 쿡북: [C4](A.ing_Transformer_Cookbook.md#c4) · [C5](A.ing_Transformer_Cookbook.md#c5) · [C15](A.ing_Transformer_Cookbook.md#c15)


In [ ]:
import torch

torch.manual_seed(0)

N = 2
src_len = 7
src_vocab_size = 50
src_pad_idx = 0
d_model = 32
h = 4

device = torch.device("cpu")

encoder = Encoder(
    src_vocab_size=src_vocab_size,
    embed_size=d_model,
    num_layers=2,
    heads=h,
    device=device,
    forward_expansion=4,
    dropout=0.0,
    max_length=100,
).to(device)

src_token_ids = torch.randint(1, src_vocab_size, (N, src_len)).to(device)  # (N, src_len)
src_token_ids[0, -2:] = src_pad_idx  # (N, src_len)

src_padding_mask = (src_token_ids != src_pad_idx).unsqueeze(1).unsqueeze(2)  # (N, 1, 1, src_len)

enc_out = encoder(src_token_ids=src_token_ids, src_padding_mask=src_padding_mask)  # (N, src_len, d_model)
study_check_shape(enc_out.shape, (N, src_len, d_model), "enc_out", "C5/C15 / N-16–N-19")
assert torch.isfinite(enc_out).all(), "NaN/Inf detected in Encoder output"
print("[OK] Encoder output shape:", tuple(enc_out.shape))


<a id="decoder-block"></a>
## 4) DecoderBlock ↔ `DecoderBlock.forward`

Decoder의 한 블록은 다음 세 역할을 수행합니다.

1) **Masked self-attention**: target의 현재 위치와 앞쪽 위치를 참고합니다.  
2) **Cross-attention**: decoder query로 encoder의 key/value를 참고합니다.  
3) **FFN**: source 정보를 반영한 각 target 위치의 특징을 변환합니다.

**CheatSheet 대응 포인트**
- Masked self-attention 뒤의 residual 결과가 cross-attention query가 됩니다.
- `self.transformer_block`에는 cross-attention과 FFN, 두 Add & Norm이 포함됩니다.
- Cross-attention 출력의 길이는 source 길이가 아니라 target query 길이입니다.

### 사고 질문
- (why) decoder self-attention에는 반드시 causal mask가 필요할까?
- (how) cross-attention에서 Q/K/V의 출처를 코드로 정확히 짚어보자.

> 허브: [R06](../Week1/transformer_paper_guide.md#r06) · [R07](../Week1/transformer_paper_guide.md#r07) · [R08](../Week1/transformer_paper_guide.md#r08)
> 대응: [CS§3](../Week1/A.ing_Transformer_Cheat_sheet.md#cs3) · [CS§4](../Week1/A.ing_Transformer_Cheat_sheet.md#cs4) · [CS§5](../Week1/A.ing_Transformer_Cheat_sheet.md#cs5) · [C13](A.ing_Transformer_Cookbook.md#c13) · [C16](A.ing_Transformer_Cookbook.md#c16) · [C17](A.ing_Transformer_Cookbook.md#c17)


In [ ]:
# =========================================================
# 4) DecoderBlock (Masked Self-Attention + Cross-Attention)
# - 쿡북: [C13-1] · [C16-1] · [C17-1]
# =========================================================

class DecoderBlock(nn.Module):
    def __init__(self, embed_size, heads, forward_expansion, dropout, device):
        super(DecoderBlock, self).__init__()
        self.norm = nn.LayerNorm(embed_size)
        self.attention = SelfAttention(embed_size, heads=heads)
        self.transformer_block = TransformerBlock(
            embed_size, heads, dropout, forward_expansion
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, value, key, src_mask, trg_mask):
        # [CS§3 Decoder Masked Self-Attention | Step 1/3] masked self-attn (미래 토큰 차단)
        # 힌트: A.ing_Transformer_Cookbook.md [C16-1]의 코드 사용법과 Shape 흐름을 먼저 확인하세요.
        # ================================================================================
        # 코드 내에서 self.attention은 SelfAttention 객체이니 이 객체를 호출했다면 forward 함수를 실행해야 합니다.
        # 여기서는 Decoder의 첫 attention을 계산하므로 미래 target을 가리는 Masked Multi-Head Self-Attention을 사용합니다.
        # SelfAttention.forward의 인자 순서와 쿡북 [C16-1]에서 각 입력의 출처를 확인해 보세요.
        # ================================================================================

        masked_self_attention_output = self.attention(____________, ____________, ____________, ____________)  # [N-20] 현재 target 표현과 causal mask로 masked self-attention을 계산한다 → [R06] [C16-1] [CS§3]

        # [CS§5 Add & Norm | Step 2/3] Residual + LayerNorm + Dropout (query 만들기)
        residual_add = ____________ + ____________  # [N-21] masked self-attention 결과에 그 입력 표현을 더한다 → [R08] [C13-1] [CS§5]
        query = self.dropout(self.norm(residual_add))

        # [CS§3 Encoder-Decoder Attention | Step 3/3] Cross-Attention(+FFN) via TransformerBlock
        # 힌트: A.ing_Transformer_Cookbook.md [C17-1]의 코드 사용법과 Shape 흐름을 먼저 확인하세요.
        out = self.transformer_block(____________, ____________, ____________, ____________)  # [N-22] decoder query가 encoder key/value를 참고하도록 블록을 호출한다 → [R06] [C17-1] [CS§3]

        return out


### ✅ Check 4: DecoderBlock shape 테스트
(빈칸을 채운 뒤 실행)

Source와 target의 길이를 다르게 두었습니다. 출력은 어느 입력의 길이를 따라야 할까요?

> **실행 전에 한 번**: 출력 shape이 얼마일지 먼저 적어 보세요 — `(  ,  ,  )`.
> 틀렸다면 어디서 어긋났는지가 곧 배울 지점입니다.

> 허브: [R06](../Week1/transformer_paper_guide.md#r06) · [R08](../Week1/transformer_paper_guide.md#r08)
> 쿡북: [C13](A.ing_Transformer_Cookbook.md#c13) · [C16](A.ing_Transformer_Cookbook.md#c16) · [C17](A.ing_Transformer_Cookbook.md#c17)


In [ ]:
import torch

torch.manual_seed(0)

N = 2
src_len = 7
trg_len = 5
d_model = 32
h = 4

device = torch.device("cpu")

decoder_block = DecoderBlock(
    embed_size=d_model,
    heads=h,
    forward_expansion=4,
    dropout=0.0,
    device=device,
).to(device)

x = torch.randn(N, trg_len, d_model).to(device)  # (N, trg_len, d_model)
value = torch.randn(N, src_len, d_model).to(device)  # (N, src_len, d_model)
key = value  # (N, src_len, d_model)

src_mask = torch.ones(N, 1, 1, src_len).to(device)  # (N, 1, 1, src_len)

trg_mask_base = torch.tril(torch.ones(trg_len, trg_len)).to(device)  # (trg_len, trg_len)
trg_mask = trg_mask_base.expand(N, 1, trg_len, trg_len)  # (N, 1, trg_len, trg_len)

out = decoder_block(x=x, value=value, key=key, src_mask=src_mask, trg_mask=trg_mask)  # (N, trg_len, d_model)
study_check_shape(out.shape, (N, trg_len, d_model), "out", "C16/C17 / N-20–N-22")
assert torch.isfinite(out).all(), "NaN/Inf detected in DecoderBlock output"
print("[OK] DecoderBlock output shape:", tuple(out.shape))


<a id="decoder"></a>
## 5) Decoder 스택과 출력 ↔ `Decoder.forward`

- Target의 토큰·위치 임베딩을 더한 뒤 DecoderBlock을 차례로 실행합니다.
- 각 층에서 decoder 표현을 갱신하고, 최종 encoder 출력은 계속 같은 source 정보로 전달합니다.
- 마지막 `fc_out`이 모델 차원 `E`를 target 어휘 크기 `Vocab`으로 바꿉니다.
- 출력 `(N, T, Vocab)`은 **raw logits**입니다. `CrossEntropyLoss` 전에 softmax를 적용하지 않습니다.

### 사고 질문
- (why) 순서 정보가 없으면 Self-Attention은 어떤 문제가 생길까?
- (how) 학습형 position embedding과 sinusoidal의 장단점은?

> 허브: [R06](../Week1/transformer_paper_guide.md#r06) · [R09](../Week1/transformer_paper_guide.md#r09) · [R10](../Week1/transformer_paper_guide.md#r10)
> 대응: [CS§3](../Week1/A.ing_Transformer_Cheat_sheet.md#cs3) · [CS§6](../Week1/A.ing_Transformer_Cheat_sheet.md#cs6) · [CS§7](../Week1/A.ing_Transformer_Cheat_sheet.md#cs7) · [C4](A.ing_Transformer_Cookbook.md#c4) · [C5](A.ing_Transformer_Cookbook.md#c5) · [C18](A.ing_Transformer_Cookbook.md#c18)


In [ ]:
# =========================================================
# 5) Decoder (Embedding + Positional Encoding + Decoder Stack + Vocab Projection)
# - 쿡북: [C4-1] · [C5-1] · [C18-1] · [C18-2]
# =========================================================

class Decoder(nn.Module):
    def __init__(
        self,
        trg_vocab_size,
        embed_size,
        num_layers,
        heads,
        forward_expansion,
        dropout,
        device,
        max_length,
    ):
        super(Decoder, self).__init__()
        self.device = device
        self.word_embedding = nn.Embedding(trg_vocab_size, embed_size)
        self.position_embedding = nn.Embedding(max_length, embed_size)

        self.layers = nn.ModuleList(
            [
                DecoderBlock(embed_size, heads, forward_expansion, dropout, device)
                for _ in range(num_layers)
            ]
        )
        self.fc_out = nn.Linear(embed_size, trg_vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, trg_token_ids, enc_out, src_padding_mask, trg_causal_mask):
        # [CS§6 토큰·임베딩 shape | Step 1/4] trg_token_ids: (N, trg_len)
        # 힌트: A.ing_Transformer_Cookbook.md [C1-1]의 코드 사용법과 Shape 흐름을 먼저 확인하세요.
        N, trg_len = trg_token_ids.shape

        # [CS§6 Embedding + Positional Encoding | Step 2/4] positions 만들기
        # 힌트: A.ing_Transformer_Cookbook.md [C5-1]의 코드 사용법과 Shape 흐름을 먼저 확인하세요.
        position_ids = torch.arange(0, ____________).to(self.device)  # [N-23] target 입력 길이만큼 위치 번호를 만든다 → [R10] [C5-1] [CS§6]
        positions = position_ids.expand(__________, ____________)  # [N-24] target 위치 번호를 배치 전체에 공유한다 → [R10] [C5-1] [CS§6]

        # [CS§6 Embedding + Positional Encoding | Step 3/4] token_emb + pos_emb (+ dropout)
        token_embedding_d_model = self.word_embedding(trg_token_ids)
        positional_embedding_d_model = self.position_embedding(positions)
        x = self.dropout(____________ + ____________)  # [N-25] target의 토큰 벡터와 위치 벡터를 더한다 → [R10] [C5-1] [CS§6]

        # [CS§3 Decoder Stack | Step 1/1] DecoderBlock 반복
        # 힌트: A.ing_Transformer_Cookbook.md [C18-1]의 코드 사용법과 Shape 흐름을 먼저 확인하세요.
        for layer in self.layers:
            x = layer(____________, ____________, ____________, ____________, ____________)  # [N-26] 현재 decoder 표현과 encoder 출력·두 마스크를 다음 Decoder 블록에 전달한다 → [R06] [C18-1] [CS§3]

        # [CS§7 Output projection | Step 4/4] vocab logits 생성
        # 힌트: A.ing_Transformer_Cookbook.md [C18-2]의 코드 사용법과 Shape 흐름을 먼저 확인하세요.
        out = self.fc_out(____________)  # [N-27] 마지막 Decoder 블록의 출력을 어휘별 점수로 바꾼다 → [R09] [C18-2] [CS§7]

        return out


<a id="transformer"></a>
## 6) Transformer 전체 구조 ↔ 마스크 생성과 `Transformer.forward`

### 핵심 규칙

- **Source mask**: PAD인 key 위치를 차단합니다. Shape는 `(N, 1, 1, S)`입니다.
- **Target causal mask**: 대각선과 아래쪽이 1인 **하삼각 허용 마스크**입니다. 위쪽의 미래 key 위치는 0으로 차단합니다. Shape는 `(N, 1, T, T)`입니다.
- Encoder에 source mask를 전달하고, Decoder에는 encoder 출력과 두 마스크를 함께 전달합니다.
- Source가 전부 PAD인 입력 등 모든 key가 차단되는 경우는 다루지 않습니다. Target은 오른쪽 PAD를 쓰고 PAD 정답을 loss에서 제외합니다.

### 사고 질문
- (why) 허용하지 않는 위치의 점수를 softmax 전에 매우 작은 값으로 바꾸는 이유는 무엇일까요?
- (how) target mask의 shape를 head/배치 차원까지 맞추는 흐름을 추적해보자.

> 허브: [R02](../Week1/transformer_paper_guide.md#r02) · [R07](../Week1/transformer_paper_guide.md#r07)
> 대응: [CS§0](../Week1/A.ing_Transformer_Cheat_sheet.md#cs0) · [CS§4](../Week1/A.ing_Transformer_Cheat_sheet.md#cs4) · [C3](A.ing_Transformer_Cookbook.md#c3) · [C19](A.ing_Transformer_Cookbook.md#c19)


In [ ]:
# =========================================================
# 6) Transformer (Encoder + Decoder + Mask 2종)
# - 쿡북: [C3-1] · [C3-2] · [C19-1]
# =========================================================
class Transformer(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        trg_vocab_size,
        src_pad_idx,
        trg_pad_idx,
        embed_size=512,
        num_layers=6,
        forward_expansion=4,
        heads=8,
        dropout=0,
        device="cpu",
        max_length=100,
    ):

        super(Transformer, self).__init__()

        self.encoder = Encoder(
            src_vocab_size,
            embed_size,
            num_layers,
            heads,
            device,
            forward_expansion,
            dropout,
            max_length,
        )

        self.decoder = Decoder(
            trg_vocab_size,
            embed_size,
            num_layers,
            heads,
            forward_expansion,
            dropout,
            device,
            max_length,
        )

        self.src_pad_idx = src_pad_idx
        self.trg_pad_idx = trg_pad_idx
        self.device = device

    def make_src_mask(self, src_token_ids):
        # [CS§4 Mask | Step 1/2] src padding mask: (src != pad) -> unsqueeze(1)->unsqueeze(2)
        # 힌트: A.ing_Transformer_Cookbook.md [C3-1]의 코드 사용법과 Shape 흐름을 먼저 확인하세요.
        # ================================================================================
        # PAD 비교가 헷갈린다면 변수 이름부터 읽어보세요.
        # "src_is_not_pad"는 src(원본)에서 is_not_pad(패딩이 아닌 부분)입니다. 그렇다면 src_token과 "무엇"이 같지 않은 부분을 확인해야 할까요?
        # ================================================================================
        src_is_not_pad = (src_token_ids != ____________)  # [N-28] source에서 PAD가 아닌 토큰 위치를 표시한다 → [R07] [C3-1] [CS§4]
        src_padding_mask = src_is_not_pad.unsqueeze(____)  # [N-29] source 유효 위치에 head용 크기 1인 축을 추가한다 → [R07] [C3-1] [CS§4]
        src_padding_mask = src_padding_mask.unsqueeze(____)  # [N-30] 모든 query가 같은 source 유효 위치를 보도록 크기 1인 축을 추가한다 → [R07] [C3-1] [CS§4]
        return src_padding_mask.to(self.device)

    def make_trg_mask(self, trg_token_ids):
        # [CS§4 Mask | Step 2/2] trg causal mask: tril(ones(L,L)).expand(N, 1, L, L)
        # 힌트: A.ing_Transformer_Cookbook.md [C3-2]의 코드 사용법과 Shape 흐름을 먼저 확인하세요.
        N, trg_len = trg_token_ids.shape
        trg_ones = torch.ones((__________, ____________))  # [N-31] target 입력 길이로 정사각형 점수표의 기본 마스크를 만든다 → [R07] [C3-2] [CS§4]
        trg_lower_triangular = torch.tril(trg_ones)  # 설명: 하삼각(미래 토큰 차단)
        trg_causal_mask = trg_lower_triangular.expand(__________, 1, ____________, ____________)  # [N-32] 하삼각 허용 마스크를 배치·head 축에 맞춰 확장한다 → [R07] [C3-2] [CS§4]

        return trg_causal_mask.to(self.device)

    def forward(self, src_token_ids, trg_token_ids):
        # [CS§0 전체 구조 | Step 1/3] mask 만들기
        # 힌트: A.ing_Transformer_Cookbook.md [C19-1]의 코드 사용법과 Shape 흐름을 먼저 확인하세요.
        src_padding_mask = self.make_src_mask(____________)  # [N-33] source 토큰으로 source padding mask를 만든다 → [R02] [C19-1] [CS§0]
        trg_causal_mask = self.make_trg_mask(____________)  # [N-34] decoder 입력 토큰으로 target causal mask를 만든다 → [R02] [C19-1] [CS§0]

        # [CS§0 전체 구조 | Step 2/3] Encoder
        # ================================================================================
        # 객체를 호출하면 객체 안의 어떤 함수가 자동으로 호출된다고 정말 많이 남겨뒀습니다.
        # 그러면 이제 찾아봅시다!
        # ================================================================================
        enc_src = self.encoder(____________, ____________)  # [N-35] source 토큰과 source 마스크를 Encoder에 전달한다 → [R02] [C19-1] [CS§0]
        # [CS§0 전체 구조 | Step 3/3] Decoder
        out = self.decoder(____________, ____________, ____________, ____________)  # [N-36] decoder 입력·encoder 출력·두 마스크를 Decoder에 전달한다 → [R02] [C19-1] [CS§0]

        return out


### ✅ Check 5.5: 마스크만 따로 확인 (중간 점검)
(빈칸을 채운 뒤 실행)

Transformer의 마스크 함수 [N-28]~[N-32]만 먼저 채웠다면 실행할 수 있습니다. Encoder/Decoder를 생성하지 않습니다. Source PAD와 target 미래 위치가 각각 어느 축에서 차단되는지, target 대각선은 허용되는지 예측하세요.

> **실행 전에 한 번**: 출력 shape이 얼마일지 먼저 적어 보세요 — source / target mask 각각 `(  ,  ,  ,  )`.
> 틀렸다면 어디서 어긋났는지가 곧 배울 지점입니다.

> 허브: [R07](../Week1/transformer_paper_guide.md#r07)
> 쿡북: [C3](A.ing_Transformer_Cookbook.md#c3)


In [ ]:
mask_only = Transformer.__new__(Transformer)
nn.Module.__init__(mask_only)
mask_only.src_pad_idx = 0
mask_only.trg_pad_idx = 0
mask_only.device = "cpu"
src_demo = torch.tensor([[1, 4, 0], [1, 5, 2]])
trg_demo = torch.tensor([[1, 4, 2], [1, 6, 2]])
src_mask_demo = mask_only.make_src_mask(src_demo)
trg_mask_demo = mask_only.make_trg_mask(trg_demo)
study_check_shape(src_mask_demo.shape, (2, 1, 1, 3), "source mask", "C3 / N-28–N-30")
study_check_shape(trg_mask_demo.shape, (2, 1, 3, 3), "target mask", "C3 / N-31–N-32")
study_check(src_mask_demo[0, 0, 0].tolist() == [True, True, False],
            "source PAD 위치를 잘못 가렸습니다", "C3 / N-28–N-30")
for row in range(3):
    for col in range(3):
        study_check(bool(trg_mask_demo[0, 0, row, col]) == (col <= row),
                    f"causal mask ({row}, {col}) 위치가 잘못됐습니다", "C3 / N-31–N-32")
print("[OK] Mask만 먼저 확인했습니다")


### ✅ Check 6: Transformer forward / mask shape 테스트
(빈칸을 채운 뒤 실행)

Target을 한 칸 이동한 입력으로 전체 모델을 호출합니다. 두 마스크의 shape와 logits의 길이를 함께 확인합니다.

> **실행 전에 한 번**: 출력 shape이 얼마일지 먼저 적어 보세요 — `(  ,  ,  )`.
> 틀렸다면 어디서 어긋났는지가 곧 배울 지점입니다.

> 허브: [R02](../Week1/transformer_paper_guide.md#r02) · [R07](../Week1/transformer_paper_guide.md#r07) · [R09](../Week1/transformer_paper_guide.md#r09)
> 쿡북: [C3](A.ing_Transformer_Cookbook.md#c3) · [C18](A.ing_Transformer_Cookbook.md#c18) · [C19](A.ing_Transformer_Cookbook.md#c19)


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

N = 2
src_len = 8
trg_len = 6

src_vocab_size = 100
trg_vocab_size = 120

src_pad_idx = 0
trg_pad_idx = 0

device = torch.device("cpu")

model = Transformer(
    src_vocab_size=src_vocab_size,
    trg_vocab_size=trg_vocab_size,
    src_pad_idx=src_pad_idx,
    trg_pad_idx=trg_pad_idx,
    embed_size=32,
    num_layers=2,
    forward_expansion=4,
    heads=4,
    dropout=0.0,
    device=device,
    max_length=100,
).to(device)

src_token_ids = torch.randint(1, src_vocab_size, (N, src_len)).to(device)  # (N, src_len)
trg_token_ids = torch.randint(1, trg_vocab_size, (N, trg_len)).to(device)  # (N, trg_len)

src_token_ids[0, -2:] = src_pad_idx  # (N, src_len)
trg_token_ids[0, -1] = trg_pad_idx  # (N, trg_len)

trg_input_ids = trg_token_ids[:, :-1]  # (N, trg_len-1)
trg_target_ids = trg_token_ids[:, 1:]  # (N, trg_len-1)

src_padding_mask = model.make_src_mask(src_token_ids)  # (N, 1, 1, src_len)
trg_causal_mask = model.make_trg_mask(trg_input_ids)  # (N, 1, trg_len-1, trg_len-1)

study_check_shape(src_padding_mask.shape, (N, 1, 1, src_len), "src_padding_mask", "C3/C19 / N-28–N-36")
study_check_shape(trg_causal_mask.shape, (N, 1, trg_len - 1, trg_len - 1), "trg_causal_mask", "C3/C19 / N-28–N-36")

logits = model(src_token_ids, trg_input_ids)  # (N, trg_len-1, trg_vocab_size)
study_check_shape(logits.shape, (N, trg_len - 1, trg_vocab_size), "logits", "C3/C19 / N-28–N-36")
assert torch.isfinite(logits).all(), "NaN/Inf detected in Transformer logits"

criterion = nn.CrossEntropyLoss(ignore_index=trg_pad_idx)

logits_2d = logits.reshape(-1, trg_vocab_size)  # (N*(trg_len-1), trg_vocab_size)
targets_1d = trg_target_ids.reshape(-1)  # (N*(trg_len-1),)

loss = criterion(logits_2d, targets_1d)
loss.backward()

print("[OK] Transformer forward + loss backward:", loss.detach().item())


### ✅ Check 6.5: 미래 정보 차단 테스트
(빈칸을 채운 뒤 실행)

Target 입력의 뒤쪽을 바꾸면 앞쪽 logits가 바뀔까요? Source 뒤에 PAD를 덧붙이면 기존 target logits가 바뀔까요? `eval()`로 dropout을 끄고, causal mask를 제거한 대조군과 함께 확인합니다.

> **실행 전에 한 번**: 출력 shape이 얼마일지 먼저 적어 보세요 — `(  ,  ,  )`.
> 틀렸다면 어디서 어긋났는지가 곧 배울 지점입니다.

> 허브: [R02](../Week1/transformer_paper_guide.md#r02) · [R06](../Week1/transformer_paper_guide.md#r06) · [R07](../Week1/transformer_paper_guide.md#r07)
> 쿡북: [C3](A.ing_Transformer_Cookbook.md#c3) · [C15](A.ing_Transformer_Cookbook.md#c15) · [C16](A.ing_Transformer_Cookbook.md#c16) · [C17](A.ing_Transformer_Cookbook.md#c17) · [C19](A.ing_Transformer_Cookbook.md#c19)


In [ ]:
m = study_small_model(layers=2).eval()
src = torch.tensor([[1, 4, 5, 2], [1, 7, 8, 2]])
target = torch.tensor([[1, 9, 10, 2], [1, 11, 12, 2]])
changed_target = target.clone()
changed_target[:, 2:] = torch.tensor([[15, 16], [17, 18]])
with torch.no_grad():
    base = m(src, target)
    altered = m(src, changed_target)
    study_check(torch.allclose(base[:, :2], altered[:, :2], atol=1e-5),
                "미래 target이 이전 logits에 영향을 줍니다", "C3/C16/C19 / N-20/N-31–N-36")
    padded_src = torch.nn.functional.pad(src, (0, 2), value=0)
    study_check(torch.allclose(base, m(padded_src, target), atol=1e-5),
                "source 뒤 PAD를 추가하자 logits가 바뀌었습니다", "C3/C15/C17 / N-19/N-22/N-28–N-30")
    # 원인이 mask임을 확인하는 대조군: 진단용 복사본만 미래를 허용합니다.
    import copy
    no_causal = copy.deepcopy(m)
    no_causal.make_trg_mask = lambda t: torch.ones(t.size(0), 1, t.size(1), t.size(1), dtype=torch.bool)
    leak = (no_causal(src, target)[:, :2] - no_causal(src, changed_target)[:, :2]).abs().max().item()
    study_check(leak > 1e-5, "대조군에서도 변화가 없어 검사가 구분력을 갖지 못합니다", "C16/C17 / N-20–N-22")
print(f"[OK] 미래 차단 / source PAD 불변성. 미래 허용 대조군 최대 차이={leak:.6f}")


### 전체 연결 실행 예시 (빈칸 없음)

두 문장을 한 배치로 넣고 target을 한 칸 이동해 호출합니다. 모델은 Transformer 클래스의 기본 설정을 사용합니다.

In [ ]:
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(device)

    x = torch.tensor([[1, 5, 6, 4, 3, 9, 5, 2, 0], [1, 8, 7, 3, 4, 5, 6, 7, 2]]).to(
        device
    )
    # (N, src_len)
    trg = torch.tensor([[1, 7, 4, 3, 5, 9, 2, 0], [1, 5, 6, 2, 4, 7, 6, 2]]).to(device)
    # (N, trg_len)

    src_pad_idx = 0
    trg_pad_idx = 0
    src_vocab_size = 10
    trg_vocab_size = 10
    model = Transformer(
        src_vocab_size,
        trg_vocab_size,
        src_pad_idx,
        trg_pad_idx,
        device=device,
    ).to(device)

    trg_input_ids = trg[:, :-1]
    # (N, trg_len-1)
    out = model(x, trg_input_ids)
    # (N, trg_len-1, trg_vocab_size)
    print(out.shape)


## 7) Train/Eval 루프 ↔ 다음 토큰 예측과 파라미터 갱신

### 핵심 포인트
- 학습: `model.train()` → gradient 초기화 → logits → loss → 역전파 → optimizer 갱신
- 평가: `model.eval()` + `torch.no_grad()`로 dropout과 gradient 기록을 끕니다.
- Target 입력·정답을 한 칸 어긋나게 준비하고, `CrossEntropyLoss`에는 raw logits를 넣습니다.
- PAD 정답은 `ignore_index`로 제외합니다. 마스크를 만드는 것과 loss에서 PAD를 제외하는 것은 별도입니다.

아래는 빈칸 없이 제공하는 작은 학습 예제입니다. 실제 데이터용 `train_one_epoch`, `evaluate`는 다음 Multi30k 실습에 있습니다.

> 허브: [R09](../Week1/transformer_paper_guide.md#r09) · [R12](../Week1/transformer_paper_guide.md#r12)
> 쿡북: [C18](A.ing_Transformer_Cookbook.md#c18) · [C19](A.ing_Transformer_Cookbook.md#c19)

### ✅ Check 7: 더미 배치로 1 step 학습 테스트
(빈칸을 채운 뒤 실행)

데이터 다운로드 없이 학습 루프를 확인합니다. Target 입력은 `trg[:, :-1]`, 정답은 `trg[:, 1:]`로 나눕니다. 어휘 축을 유지한 logits와 정답의 위치 수를 맞춰 loss를 계산합니다.

> **실행 전에 한 번**: 출력 shape이 얼마일지 먼저 적어 보세요 — `(  ,  ,  )`.
> 틀렸다면 어디서 어긋났는지가 곧 배울 지점입니다.

> 허브: [R09](../Week1/transformer_paper_guide.md#r09) · [R12](../Week1/transformer_paper_guide.md#r12)
> 쿡북: [C18](A.ing_Transformer_Cookbook.md#c18) · [C19](A.ing_Transformer_Cookbook.md#c19)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence

device = "cuda" if torch.cuda.is_available() else "cpu"

# --- tiny toy vocab ---
src_vocab_size = 50
trg_vocab_size = 60
src_pad_idx = 0
trg_pad_idx = 0

# model hyperparams (paper-base 느낌, but tiny for smoke test)
model = Transformer(
    src_vocab_size=src_vocab_size,
    trg_vocab_size=trg_vocab_size,
    src_pad_idx=src_pad_idx,
    trg_pad_idx=trg_pad_idx,
    embed_size=128,
    num_layers=2,
    forward_expansion=4,
    heads=4,
    dropout=0.1,
    device=device,
    max_length=64,
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=trg_pad_idx)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# --- dummy batch (variable lengths + padding) ---
src_batch = [
    torch.tensor([1, 5, 6, 4, 3, 9, 2]),
    torch.tensor([1, 8, 7, 3, 4, 5]),
]
trg_batch = [
    torch.tensor([1, 7, 4, 3, 5, 9, 2, 2]),
    torch.tensor([1, 5, 6, 2, 4, 7]),
]

src = pad_sequence(src_batch, batch_first=True, padding_value=src_pad_idx).to(device)
# (N, src_len)
trg = pad_sequence(trg_batch, batch_first=True, padding_value=trg_pad_idx).to(device)
# (N, trg_len)

trg_input = trg[:, :-1]
# (N, trg_len-1)
trg_y = trg[:, 1:]
# (N, trg_len-1)

logits = model(src, trg_input)
# (N, trg_len-1, trg_vocab_size)

logits_flat = logits.reshape(-1, logits.size(-1))
# (N*(trg_len-1), trg_vocab_size)
trg_y_flat = trg_y.reshape(-1)
# (N*(trg_len-1),)

optimizer.zero_grad()
loss = criterion(logits_flat, trg_y_flat)
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
optimizer.step()

print("✅ smoke loss:", float(loss.item()))


### ✅ Check 7.5: 작은 문장 집합 과적합 테스트
(빈칸을 채운 뒤 실행)

Source 순서를 뒤집는 인공 과제의 같은 4문장을 120번 학습합니다. CPU·고정 seed·dropout=0으로 확인합니다. 먼저 loss가 어떤 방향으로 변할지 적어 보세요. 통과하더라도 앞의 마스크·cross-attention 검사를 대신하거나 번역 품질을 보장하지는 않습니다.

> **실행 전에 한 번**: 출력 shape이 얼마일지 먼저 적어 보세요 — `(  ,  ,  )`.
> 틀렸다면 어디서 어긋났는지가 곧 배울 지점입니다.

> 허브: [R02](../Week1/transformer_paper_guide.md#r02) · [R09](../Week1/transformer_paper_guide.md#r09) · [R12](../Week1/transformer_paper_guide.md#r12)
> 쿡북: [C18](A.ing_Transformer_Cookbook.md#c18) · [C19](A.ing_Transformer_Cookbook.md#c19)


In [ ]:
tiny_model = study_small_model(seed=23)
tiny_src = torch.tensor([[1,4,5,2], [1,6,7,2], [1,8,9,2], [1,10,11,2]])
tiny_target = torch.tensor([[1,5,4,2], [1,7,6,2], [1,9,8,2], [1,11,10,2]])
tiny_in, tiny_y = tiny_target[:, :-1], tiny_target[:, 1:]
tiny_optimizer = torch.optim.Adam(tiny_model.parameters(), lr=0.01)
tiny_loss_fn = nn.CrossEntropyLoss(ignore_index=0)
tiny_losses = []
tiny_model.train()
for step in range(120):
    tiny_optimizer.zero_grad()
    scores = tiny_model(tiny_src, tiny_in)
    loss = tiny_loss_fn(scores.flatten(0, 1), tiny_y.flatten())
    study_check(torch.isfinite(loss), "작은 데이터 loss에 NaN/Inf가 있습니다", "R12 / 학습률·mask 확인")
    loss.backward()
    torch.nn.utils.clip_grad_norm_(tiny_model.parameters(), 1.0)
    tiny_optimizer.step()
    tiny_losses.append(loss.item())
tiny_model.eval()
with torch.no_grad():
    predicted = tiny_model(tiny_src, tiny_in).argmax(-1)
    accuracy = (predicted == tiny_y).float().mean().item()
study_check(tiny_losses[-1] < tiny_losses[0] * 0.2 and accuracy >= 0.95,
            f"작은 데이터 암기 실패: loss {tiny_losses[0]:.3f}→{tiny_losses[-1]:.3f}, acc={accuracy:.3f}",
            "R02/R09/R12 / shift·raw logits·optimizer·residual 확인")
print(f"[OK] 작은 데이터 학습: loss {tiny_losses[0]:.3f}→{tiny_losses[-1]:.3f}, token acc={accuracy:.3f}")


## 8) 실습: Multi30k 학습을 실제로 돌려보기

아래 코드는 **이 노트북에서 만든 Transformer를 사용하는 Multi30k(독일어→영어) 학습 스켈레톤**입니다.

- 이 섹션은 **빈칸이 없습니다.** 위의 빈칸 구현을 먼저 완료하세요.
- 처음에는 `num_epochs=1`로 동작을 확인합니다. 실행 시간은 장치와 데이터 처리 속도에 따라 달라집니다.
- `Transformer`를 이미 위에서 정의했으므로 별도 모델 파일을 import할 필요가 없습니다.
- Multi30k는 논문의 WMT14와 데이터·토큰화·평가 조건이 다릅니다. 여기서 얻은 BLEU를 논문 수치와 직접 비교하지 않습니다.

### 훈련 설정
- Adam: `β1=0.9`, `β2=0.98`, `ε=1e-9`
- Learning rate: warmup 4,000 step 이후 inverse square-root decay
- Dropout `0.1`, 학습 label smoothing `0.1`, PAD 정답 제외
- 학습형 위치 임베딩, 단어 단위 토큰화, greedy decoding을 사용합니다.

### 필요한 패키지 (처음 1회)

아래 설치 셀과 학습 셀은 **Multi30k 실습을 할 때만** 실행합니다. 뒤의 E01/E02는 이 설치나 학습 없이 실행할 수 있습니다.

> 허브: [R12](../Week1/transformer_paper_guide.md#r12) · [R13](../Week1/transformer_paper_guide.md#r13)
> 쿡북: [C18](A.ing_Transformer_Cookbook.md#c18) · [C19](A.ing_Transformer_Cookbook.md#c19)

평가 loss는 label smoothing을 적용하지 않은 non-PAD 토큰당 cross entropy입니다. 영어를 소문자로 학습하므로 BLEU도 대소문자를 구분하지 않습니다. `max_len`은 BOS/EOS를 포함한 학습 입력 길이, `max_new_tokens`는 BOS 뒤에 생성할 토큰 수의 상한입니다. 생성 입력은 위치 embedding의 용량 안에서 유지됩니다.


In [ ]:
!pip install datasets sacrebleu spacy
!python -m spacy download de_core_news_sm
!python -m spacy download en_core_web_sm


> ✅ 처음에는 `num_epochs=1`로 동작을 확인한 뒤 epochs를 늘리세요. 모델 기본값은 `d_model=512`, 6개 층이며, 장치 메모리가 부족하면 `d_model=128, num_layers=2, num_heads=4`처럼 줄여 실행할 수 있습니다.


In [ ]:
# Multi30k 독일어→영어 학습
# 이 노트북의 Transformer와 spaCy 단어 단위 토큰화를 사용합니다.
# 학습: label smoothing / 평가: non-PAD 토큰당 일반 cross entropy

import math
import random
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

import spacy

# --- optional: BLEU (sacrebleu) ---
try:
    import sacrebleu
except Exception:
    sacrebleu = None

# --- HF datasets for Multi30k ---
try:
    from datasets import load_dataset
except Exception:
    load_dataset = None


SPECIAL_TOKENS = {
    "pad": "<pad>",
    "unk": "<unk>",
    "sos": "<sos>",
    "eos": "<eos>",
}


def set_seed(seed: int = 42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def build_vocab(sentences, tokenize_fn, max_size: int = 10000, min_freq: int = 2):
    """Word-level vocab builder (toy / study-friendly)."""
    counter = Counter()
    for s in sentences:
        counter.update(tokenize_fn(s))

    # reserve specials at the beginning so indices are stable
    itos = [
        SPECIAL_TOKENS["pad"],
        SPECIAL_TOKENS["unk"],
        SPECIAL_TOKENS["sos"],
        SPECIAL_TOKENS["eos"],
    ]

    for tok, freq in counter.most_common():
        if freq < min_freq:
            continue
        if tok in itos:
            continue
        itos.append(tok)
        if len(itos) >= max_size:
            break

    stoi = {tok: i for i, tok in enumerate(itos)}
    return stoi, itos


class Multi30kWordDataset(Dataset):
    """(de, en) sentence pairs -> (src_ids, trg_ids)"""

    def __init__(
        self,
        split,
        src_tokenize_fn,
        trg_tokenize_fn,
        src_stoi,
        trg_stoi,
        src_max_len: int = 100,
        trg_max_len: int = 100,
    ):
        self.split = split
        self.src_tokenize_fn = src_tokenize_fn
        self.trg_tokenize_fn = trg_tokenize_fn
        self.src_stoi = src_stoi
        self.trg_stoi = trg_stoi
        self.src_max_len = src_max_len
        self.trg_max_len = trg_max_len

        self.src_unk = src_stoi[SPECIAL_TOKENS["unk"]]
        self.trg_unk = trg_stoi[SPECIAL_TOKENS["unk"]]
        self.src_sos = src_stoi[SPECIAL_TOKENS["sos"]]
        self.src_eos = src_stoi[SPECIAL_TOKENS["eos"]]
        self.trg_sos = trg_stoi[SPECIAL_TOKENS["sos"]]
        self.trg_eos = trg_stoi[SPECIAL_TOKENS["eos"]]

    def __len__(self):
        return len(self.split)

    def __getitem__(self, idx):
        example = self.split[idx]
        src_text = example["de"]
        trg_text = example["en"]

        src_tokens = self.src_tokenize_fn(src_text)[: self.src_max_len - 2]
        trg_tokens = self.trg_tokenize_fn(trg_text)[: self.trg_max_len - 2]

        src_ids = [self.src_sos] + [self.src_stoi.get(t, self.src_unk) for t in src_tokens] + [
            self.src_eos
        ]
        trg_ids = [self.trg_sos] + [self.trg_stoi.get(t, self.trg_unk) for t in trg_tokens] + [
            self.trg_eos
        ]

        return torch.tensor(src_ids, dtype=torch.long), torch.tensor(trg_ids, dtype=torch.long)


def make_collate_fn(src_pad_idx: int, trg_pad_idx: int):
    def collate_fn(batch):
        src_list = [b[0] for b in batch]
        trg_list = [b[1] for b in batch]
        src = pad_sequence(src_list, batch_first=True, padding_value=src_pad_idx)
        # (N, src_len)
        trg = pad_sequence(trg_list, batch_first=True, padding_value=trg_pad_idx)
        # (N, trg_len)
        return src, trg

    return collate_fn


def noam_lr_lambda(step: int, d_model: int, warmup_steps: int = 4000):
    """Paper: lr = d_model^{-0.5} * min(step^{-0.5}, step * warmup^{-1.5})"""
    step = max(step, 1)
    return (d_model ** -0.5) * min(step ** -0.5, step * (warmup_steps ** -1.5))


@torch.no_grad()
def greedy_decode(
    model,
    src_ids_1d: torch.Tensor,
    src_pad_idx: int,
    trg_sos_idx: int,
    trg_eos_idx: int,
    max_new_tokens: int,
    device: str,
):
    """Greedy decoding for quick sanity check (not beam search)."""
    model.eval()

    src = src_ids_1d.unsqueeze(0).to(device)
    # (N=1, src_len)

    generated = [trg_sos_idx]

    if type(max_new_tokens) is not int or max_new_tokens < 1:
        raise ValueError("max_new_tokens는 양의 정수여야 합니다.")
    position_capacity = model.decoder.position_embedding.num_embeddings
    # BOS를 포함한 입력이 위치 표 용량을 넘기 전에 생성을 끝냅니다.
    for _ in range(min(max_new_tokens, position_capacity)):
        trg = torch.tensor(generated, dtype=torch.long, device=device).unsqueeze(0)
        # (N=1, trg_len)

        logits = model(src, trg)
        # (N=1, trg_len, trg_vocab_size)

        next_token = int(logits[0, -1].argmax(dim=-1).item())
        generated.append(next_token)

        if next_token == trg_eos_idx:
            break

    return generated


def train_one_epoch(model, loader, optimizer, scheduler, criterion, device: str):
    model.train()
    total_loss = 0.0
    total_tokens = 0

    for src, trg in loader:
        src = src.to(device)
        # (N, src_len)
        trg = trg.to(device)
        # (N, trg_len)

        # --- Teacher forcing shift (표준 패턴) ---
        trg_input = trg[:, :-1]
        # (N, trg_len-1)
        trg_y = trg[:, 1:]
        # (N, trg_len-1)

        logits = model(src, trg_input)
        # (N, trg_len-1, trg_vocab_size)

        logits_flat = logits.reshape(-1, logits.size(-1))
        # (N*(trg_len-1), trg_vocab_size)
        trg_y_flat = trg_y.reshape(-1)
        # (N*(trg_len-1),)

        optimizer.zero_grad()
        loss = criterion(logits_flat, trg_y_flat)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        tokens = int(trg_y_flat.ne(criterion.ignore_index).sum())
        tokens = int(trg_y.ne(criterion.ignore_index).sum())
        total_loss += float(loss.item()) * tokens
        total_tokens += tokens * tokens
        total_tokens += tokens

    return total_loss / max(1, total_tokens)


@torch.no_grad()
def evaluate(model, loader, criterion, device: str):
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    for src, trg in loader:
        src = src.to(device)
        trg = trg.to(device)

        trg_input = trg[:, :-1]
        trg_y = trg[:, 1:]

        logits = model(src, trg_input)

        logits_flat = logits.reshape(-1, logits.size(-1))
        trg_y_flat = trg_y.reshape(-1)

        loss = criterion(logits_flat, trg_y_flat)
        tokens = int(trg_y.ne(criterion.ignore_index).sum())
        total_loss += float(loss.item()) * tokens
        total_tokens += tokens

    return total_loss / max(1, total_tokens)


def main(
    num_epochs: int = 1,
    batch_size: int = 64,
    max_vocab_size: int = 10000,
    min_freq: int = 2,
    max_len: int = 100,
    d_model: int = 512,
    num_layers: int = 6,
    num_heads: int = 8,
    forward_expansion: int = 4,
    dropout: float = 0.1,
    warmup_steps: int = 4000,
    max_new_tokens: int = 50,
):
    if type(max_len) is not int or max_len < 3:
        raise ValueError("max_len은 BOS/EOS와 내용 토큰을 담을 수 있는 3 이상의 정수여야 합니다.")
    if load_dataset is None:
        raise ImportError("❌ datasets가 없습니다. 먼저 `pip install datasets`를 실행하세요.")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    set_seed(42)

    # --- Load Multi30k (bentrevett subset on HF hub) ---
    raw = load_dataset("bentrevett/multi30k")
    train_raw = raw["train"]
    valid_raw = raw["validation"]
    test_raw = raw["test"]

    # --- Tokenizers (spaCy) ---
    # spaCy v3+에서는 'de'/'en' shortcut이 아니라 full model name이 필요합니다.
    spacy_de = spacy.load("de_core_news_sm")
    spacy_en = spacy.load("en_core_web_sm")

    def tokenize_de(text: str):
        return [tok.text.lower() for tok in spacy_de.tokenizer(text)]

    def tokenize_en(text: str):
        return [tok.text.lower() for tok in spacy_en.tokenizer(text)]

    # --- Build vocab (word-level) ---
    src_stoi, src_itos = build_vocab(train_raw["de"], tokenize_de, max_size=max_vocab_size, min_freq=min_freq)
    trg_stoi, trg_itos = build_vocab(train_raw["en"], tokenize_en, max_size=max_vocab_size, min_freq=min_freq)

    src_pad_idx = src_stoi[SPECIAL_TOKENS["pad"]]
    trg_pad_idx = trg_stoi[SPECIAL_TOKENS["pad"]]
    trg_sos_idx = trg_stoi[SPECIAL_TOKENS["sos"]]
    trg_eos_idx = trg_stoi[SPECIAL_TOKENS["eos"]]

    # --- Datasets / Loaders ---
    train_ds = Multi30kWordDataset(train_raw, tokenize_de, tokenize_en, src_stoi, trg_stoi, max_len, max_len)
    valid_ds = Multi30kWordDataset(valid_raw, tokenize_de, tokenize_en, src_stoi, trg_stoi, max_len, max_len)
    test_ds = Multi30kWordDataset(test_raw, tokenize_de, tokenize_en, src_stoi, trg_stoi, max_len, max_len)

    collate_fn = make_collate_fn(src_pad_idx, trg_pad_idx)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    valid_loader = DataLoader(valid_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    # --- Model (this notebook's from-scratch Transformer) ---
    model = Transformer(
        src_vocab_size=len(src_itos),
        trg_vocab_size=len(trg_itos),
        src_pad_idx=src_pad_idx,
        trg_pad_idx=trg_pad_idx,
        embed_size=d_model,
        num_layers=num_layers,
        forward_expansion=forward_expansion,
        heads=num_heads,
        dropout=dropout,
        device=device,
        max_length=max_len,
    ).to(device)

    # --- Paper-aligned optimizer + schedule ---
    optimizer = optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)

    scheduler = optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=lambda step: noam_lr_lambda(step + 1, d_model=d_model, warmup_steps=warmup_steps),
    )

    criterion = nn.CrossEntropyLoss(
        ignore_index=trg_pad_idx,
        label_smoothing=0.1,  # paper: ε_ls = 0.1
    )

    eval_criterion = nn.CrossEntropyLoss(ignore_index=trg_pad_idx)

    # --- Train ---
    for epoch in range(num_epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, criterion, device)
        valid_loss = evaluate(model, valid_loader, eval_criterion, device)

        print(f"epoch={epoch:02d} train_loss={train_loss:.4f} valid_loss={valid_loss:.4f}")

        # quick qualitative check: translate a random validation sample
        sample = valid_raw[random.randrange(len(valid_raw))]
        src_text = sample["de"]
        trg_text = sample["en"]

        src_tokens = tokenize_de(src_text)[: max_len - 2]
        src_ids = [src_stoi[SPECIAL_TOKENS["sos"]]] + [src_stoi.get(t, src_stoi[SPECIAL_TOKENS["unk"]]) for t in src_tokens] + [src_stoi[SPECIAL_TOKENS["eos"]]]
        src_ids = torch.tensor(src_ids, dtype=torch.long)

        pred_ids = greedy_decode(
            model=model,
            src_ids_1d=src_ids,
            src_pad_idx=src_pad_idx,
            trg_sos_idx=trg_sos_idx,
            trg_eos_idx=trg_eos_idx,
            max_new_tokens=max_new_tokens,
            device=device,
        )

        pred_tokens = [trg_itos[i] for i in pred_ids]
        pred_tokens = [t for t in pred_tokens if t not in {SPECIAL_TOKENS["sos"], SPECIAL_TOKENS["eos"], SPECIAL_TOKENS["pad"]}]

        print("DE:", src_text)
        print("GT:", trg_text)
        print("PR:", " ".join(pred_tokens))
        print("-" * 80)

    # --- BLEU (optional) ---
    if sacrebleu is None:
        print("sacrebleu가 없어서 BLEU를 생략합니다. (pip install sacrebleu)")
        return

    # quick BLEU on a small subset (speed)
    model.eval()
    preds = []
    refs = []

    for i in range(min(200, len(test_raw))):
        ex = test_raw[i]
        src_text = ex["de"]
        ref_text = ex["en"]

        src_tokens = tokenize_de(src_text)[: max_len - 2]
        src_ids = [src_stoi[SPECIAL_TOKENS["sos"]]] + [src_stoi.get(t, src_stoi[SPECIAL_TOKENS["unk"]]) for t in src_tokens] + [src_stoi[SPECIAL_TOKENS["eos"]]]
        src_ids = torch.tensor(src_ids, dtype=torch.long)

        pred_ids = greedy_decode(
            model=model,
            src_ids_1d=src_ids,
            src_pad_idx=src_pad_idx,
            trg_sos_idx=trg_sos_idx,
            trg_eos_idx=trg_eos_idx,
            max_new_tokens=max_new_tokens,
            device=device,
        )

        pred_tokens = [trg_itos[i] for i in pred_ids]
        pred_tokens = [t for t in pred_tokens if t not in {SPECIAL_TOKENS["sos"], SPECIAL_TOKENS["eos"], SPECIAL_TOKENS["pad"]}]
        preds.append(" ".join(pred_tokens))
        refs.append(ref_text)

    metric = sacrebleu.metrics.BLEU(lowercase=True, tokenize="13a")
    bleu = metric.corpus_score(preds, [refs]).score
    print(f"BLEU (greedy, lowercase, test {len(refs)} samples) = {bleu:.2f}")
    print("BLEU signature:", metric.get_signature())
    print("Generation limit:", min(max_new_tokens, model.decoder.position_embedding.num_embeddings))


# ✅ 실행 예시 (처음엔 epochs를 줄여서!)
main(num_epochs=1)


## 9) (실험) Scaling과 위치 정보의 역할

빈칸을 채운 모델과 작은 인공 입력으로 두 설계의 역할을 관찰합니다. **Multi30k 학습을 건너뛰어도 실행할 수 있습니다.** E01은 무작위 Q·K만, E02는 학습하지 않은 작은 Encoder를 사용합니다.

> 허브: [R04](../Week1/transformer_paper_guide.md#r04) · [R10](../Week1/transformer_paper_guide.md#r10) · [R13](../Week1/transformer_paper_guide.md#r13)

### ✋ 실행 전에 먼저 예측하세요

**돌리기 전에** 아래 표의 예측과 근거를 채웁니다. 자기 예측과 관찰을 비교하는 것이 목적입니다. 실험 뒤에는 나머지 칸을 채우세요.

| 실험 | 내 예측 | 근거로 삼은 R | 고정한 조건 | 변경한 조건 | 실제 관찰 | 해석과 한계 |
| --- | --- | --- | --- | --- | --- | --- |
| E01: head 차원이 커질 때 scaling 전후의 점수 분산·확률 집중도 | | | | | | |
| E02: source 순서를 바꿀 때 위치 정보 유무에 따른 Encoder 출력 | | | | | | |

예측과 달랐다면 어느 가정을 바꿔야 할까요? 이 결과만으로 학습된 번역 모델의 성능을 판단할 수 없는 이유도 적으세요.

```text
답:
```


### E01. Scaling이 softmax 분포에 주는 영향

**실행 전:** d_k가 커질수록 scaling 전/후 점수 분산, 최대 attention 확률, 엔트로피가 어떻게 변할지 적으세요.
독립인 표준정규 Q/K를 만들고, 같은 점수에 scaling만 다르게 적용합니다. 엔트로피가 낮을수록 분포가 소수 위치에 몰립니다.
학습된 attention도 항상 이 분포를 따른다는 뜻은 아니며, 여기서는 gradient나 번역 성능을 직접 측정하지 않습니다.

> 허브: [R04](../Week1/transformer_paper_guide.md#r04) · [R13](../Week1/transformer_paper_guide.md#r13)
> 쿡북: [C10](A.ing_Transformer_Cookbook.md#c10)


In [ ]:
import math
with torch.random.fork_rng():
    torch.manual_seed(31)
    print("d_k | raw variance | scaled variance | raw/scaled max probability | raw/scaled entropy")
    for width in [4, 16, 64, 256]:
        queries = torch.randn(256, 1, width)
        keys = torch.randn(256, 16, width)
        scores = (queries * keys).sum(dim=-1)
        raw_p = scores.softmax(-1)
        scaled_scores = scores / math.sqrt(width)
        scaled_p = scaled_scores.softmax(-1)
        def entropy(p):
            return -(p * p.clamp_min(1e-12).log()).sum(-1).mean().item()
        print(f"{width:3d} | {scores.var().item():10.3f} | {scaled_scores.var().item():13.3f} | "
              f"{raw_p.max(-1).values.mean().item():.3f}/{scaled_p.max(-1).values.mean().item():.3f} | "
              f"{entropy(raw_p):.3f}/{entropy(scaled_p):.3f}")


### E02. 위치 정보를 빼면 무엇을 구별할 수 없는가

**실행 전:** source의 두 토큰 순서를 바꿉니다. 위치 embedding을 0으로 만든 encoder 출력도 동일한 순열로 바뀔까요? 위치별 출력을 평균한 벡터는 달라질까요?
같은 encoder 가중치를 복사하고 위치 embedding만 0으로 바꿉니다. causal decoder는 사용하지 않으며, 이 관찰을 전체 Transformer가 순서를 무시한다는 주장으로 확대하지 않습니다.

> 허브: [R10](../Week1/transformer_paper_guide.md#r10) · [R13](../Week1/transformer_paper_guide.md#r13)
> 쿡북: [C5](A.ing_Transformer_Cookbook.md#c5)


In [ ]:
import copy
position_model = study_small_model(seed=37).eval()
with_position = position_model.encoder
without_position = copy.deepcopy(with_position)
with torch.no_grad():
    without_position.position_embedding.weight.zero_()
sequence = torch.tensor([[1, 4, 7, 2]])
permutation = torch.tensor([0, 2, 1, 3])
source_mask = torch.ones(1, 1, 1, 4, dtype=torch.bool)
with torch.no_grad():
    for name, encoder in [("위치 없음", without_position), ("학습형 위치 있음", with_position)]:
        a = encoder(sequence, source_mask)
        b = encoder(sequence[:, permutation], source_mask)
        equivariance_error = (a[:, permutation] - b).abs().max().item()
        pooled_error = (a.mean(1) - b.mean(1)).abs().max().item()
        print(f"{name}: 같은 순열로 정렬한 출력 차이={equivariance_error:.6f}, 평균 벡터 차이={pooled_error:.6f}")
        if name == "위치 없음":
            study_check(equivariance_error < 1e-5 and pooled_error < 1e-5,
                        "위치 없는 encoder의 순열 성질이 맞지 않습니다", "C5/C7/C15 / 축·dropout 확인")


### 실행 결과 해석 — 자기 결과를 본 뒤에 펼치세요

<details>
<summary>E01/E02에서 확인할 경향</summary>

- **E01**: 독립적인 표준정규 Q·K를 사용할 때 scaling은 head 차원이 커지며 내적 점수 분산이 커지는 효과를 완화합니다. Softmax가 얼마나 집중되는지도 함께 비교하세요.
- **E02**: 위치 정보가 없는 Encoder는 입력 순열에 따라 출력도 같은 순열로 바뀝니다. 이때 위치별 출력을 평균하면 순서 차이를 구별할 수 없습니다. 위치 임베딩을 넣으면 이 대칭성이 일반적으로 깨집니다.

특정 수치를 맞히는 과제가 아닙니다. E01은 학습된 attention의 성능을, E02는 causal Decoder까지 포함한 전체 모델의 성질을 직접 검증하지 않습니다.

</details>

## 풀이 후 확인

[정답 노트북](A.ing_Transformer_from_scratch_answer.ipynb)은 완성 모델과 핵심 검사를 별도로 제공합니다. 틀린 문장은 N 번호로 비교한 뒤, 왜 그 답인지 R/C 번호를 근거로 설명하세요.
